# 4 · How the pH is obtained

The pH enters every reaction through the H⁺ concentration (R1 and R2 consume four protons per Mn). Bernard et al. (2025, GITT) compared three ways of getting it; the corrected model adds a fourth, the solution equilibria at every point:

| `ph_mode` | |
|---|---|
| `speciation` (default) | from the 21 solution equilibria and the transported totals |
| `zhs_equilibrium` | the pH at which ZHS is saturated (*R2Precipitation*) |
| `fixed` | a constant 4.8 (*R2Fixed*) |
| `spline` | the original PHREEQC surrogate pH(Zn, Mn) (*R2PHREEQC*) |

Each runs a discharge and a charge at ±100 mA/g. The `spline` case is slower (the spline is evaluated cell by cell).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from znmno2_model.params import Params
from znmno2_model.simulate import run
from znmno2_model.model import Model

def compare(runs, title, cols=("V", "pH_cath")):
    """Plot several runs (dict label -> Result) against capacity."""
    fig, ax = plt.subplots(1, len(cols), figsize=(6 * len(cols), 4.2))
    ax = np.atleast_1d(ax)
    for label, r in runs.items():
        for a, c in zip(ax, cols):
            a.plot(r.column("mAhg"), r.column(c), label=label)
    for a, c in zip(ax, cols):
        a.set(xlabel="capacity [mAh/g]  (decreasing = charge)", ylabel=c)
    ax[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def summary(runs):
    for label, r in runs.items():
        q = r.column("mAhg")
        print(f"{label:28s} exit {r.exit_reason:16s} discharged {q.max():6.1f} mAh/g, final {q[-1]:6.1f} mAh/g")

In [ ]:
runs = {mode: run(Params(ph_mode=mode, steps="cc I=100 Vmin=1.0; cc I=-100 Vmax=1.85", end_on_cutoff=False)) for mode in ("speciation", "zhs_equilibrium", "fixed", "spline")}
summary(runs)
compare(runs, "pH treatment (pH_cath is the solution's own pH in every case)");

`pH_cath` is always the solution's pH from the speciation; with the other modes the reactions see a different H⁺. The voltage shows what each treatment implies for the reactions.

## What the reactions see

The pH each mode feeds to the reactions over the discharge:

In [ ]:
from znmno2_model.model import Model
fig, ax = plt.subplots(figsize=(7, 4))
for mode in ("speciation", "zhs_equilibrium", "fixed", "spline"):
    p = Params(ph_mode=mode, steps="cc I=100 Vmin=1.0", write_interval=300.0)
    r = run(p, keep_states=True)
    m = Model(p)
    ph = [np.mean(-np.log10(m._basis(x, m.free(x))[m.cath, 0])) for t, x in r.states]
    ax.plot(r.column("mAhg"), ph, label=mode)
ax.set(xlabel="capacity [mAh/g]", ylabel="pH used by the reactions (cathode mean)"); ax.legend();